# 🧪 W5-D5 概念实验：自洽性选择器、元认知校准、反事实与迷你 ReAct

> 配套阅读：`ima/第5周-Day5-高级推理技巧与Agent推理框架.md`（自洽性/元认知/反事实/工具组合的定义在那边）
>
> 本 notebook 回答四个问题：
> 1. 多条推理链 + 多数投票 / **置信度加权投票**，比"挑最自信的一条"强多少？
> 2. 模型的"自信"和"正确"是一回事吗？（校准曲线与 ECE）
> 3. 反事实提升（"如果不做促销"）要多大样本才测得准？
> 4. 一个 40 行的迷你 ReAct 循环长什么样：工具、审计、以及"知道自己缺什么"。

## 实验 1：可审计的自洽性选择器

生成 k=8 条推理链：单链正确率 55%；错误答案**不是均匀散开**——
一半概率集中跌进同一个"诱人错误项"（单位换错、差一位数这类系统性偏差）。
每条链带自报置信度（答对的链平均更自信，但不完美）。
对比三种取答案方式：最高置信单链 / 多数投票 / 置信度加权投票。

In [ ]:
import numpy as np

rng = np.random.default_rng(13)
n_problems, k, n_wrong, p_chain = 20_000, 8, 9, 0.55

correct = rng.random((n_problems, k)) < p_chain
wrong_ans = np.where(rng.random((n_problems, k)) < 0.5, 1,
                     1 + rng.integers(0, n_wrong, (n_problems, k)))   # 诱人错误项=1
answers = np.where(correct, 0, wrong_ans)
conf = np.where(correct, rng.beta(5, 2, (n_problems, k)),      # 答对：平均~0.71
                          rng.beta(4, 3, (n_problems, k)))     # 答错：平均~0.57

def vote(answers, weights):
    """对最终答案累计权重，取最大（0 = 正确答案）。每题留下票型即可审计"""
    m = int(answers.max()) + 1
    W = np.zeros((answers.shape[0], m))
    for j in range(m):
        W[:, j] = ((answers == j) * weights).sum(axis=1)
    return W.argmax(axis=1)

maj = vote(answers, np.ones_like(conf))
wtd = vote(answers, conf)
top1 = answers[np.arange(n_problems), conf.argmax(axis=1)]

print(f"单条链基线                     {p_chain:.1%}")
print(f"挑最高置信度的那条单链          {(top1 == 0).mean():.1%}")
print(f"多数投票（{k} 条链）           {(maj == 0).mean():.1%}")
print(f"置信度加权投票（{k} 条链）      {(wtd == 0).mean():.1%}")
print()
print("投票明显优于挑单链（即使那条链'最自信'）；加权投票再补一手：")
print("当错误答案系统性聚堆（诱人错误项）时，低置信链的票被自动打折。")
print("可审计性：每题保留 8 个 (答案, 置信度, 链摘要)，出错能回放是哪条链带偏的。")

## 实验 2：元认知校准——"自信 ≠ 正确"

让两个 Agent 对 2 万个判断自报置信度：
- 校准良好型：实际正确率 = 置信度（点落在对角线上）
- 过度自信型：实际正确率只有 0.6×置信度+0.05（同样的嘴硬，兑现打折）

按置信度分箱画**可靠性图**，并计算 ECE（期望校准误差，越小越好）。

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

rng = np.random.default_rng(17)
n = 20_000
conf = np.clip(rng.beta(4, 2, n), 0.01, 0.99)     # 自报置信度，均值~0.67

def make_agent(slope, bias):
    """真实正确率 = slope*conf + bias；slope<1 即过度自信"""
    return rng.random(n) < np.clip(slope * conf + bias, 0, 1)

bins = np.linspace(0, 1, 11)
mid = (bins[:-1] + bins[1:]) / 2

plt.figure(figsize=(6.2, 4.4))
for name, ans in [("校准良好 (slope=1.0)", make_agent(1.0, 0.0)),
                  ("过度自信 (slope=0.6)", make_agent(0.6, 0.05))]:
    ece, pts = 0.0, []
    for lo, hi, m in zip(bins[:-1], bins[1:], mid):
        msk = (conf >= lo) & (conf < hi)
        if msk.sum() < 50:
            continue
        acc = ans[msk].mean()
        ece += msk.mean() * abs(acc - m)
        pts.append((m, acc))
    plt.plot([p[0] for p in pts], [p[1] for p in pts], "o-", label=f"{name}，ECE={ece:.3f}")
plt.plot([0, 1], [0, 1], "k--", lw=1, label="完美校准")
plt.xlabel("自报置信度"); plt.ylabel("实际正确率")
plt.title("元认知校准曲线：越贴近对角线，越'知道自己知道'")
plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("工程含义：Agent 的置信度只有在'校准'时才能当路由/人工升级的阈值用；")
print("先收集 (自报置信, 对错) 数据画这张图，再决定多大置信才免人工复核。")

## 实验 3：反事实提升要多大样本才测得准？

"如果没做促销会怎样"无法直接观察，只能用对照组近似。
对照组复购率 12%、促销组 13.5%（真实提升 +1.5pp）。
看 n=5000 时的 bootstrap 区间，以及不同样本量下这个提升有几个标准误。

In [ ]:
import numpy as np

rng = np.random.default_rng(2)
n = 5_000
ctrl = rng.random(n) < 0.12
treat = rng.random(n) < 0.135

uplift = treat.mean() - ctrl.mean()
bs = []
r = np.random.default_rng(3)
for _ in range(2000):
    idx = r.integers(0, n, n)
    bs.append(treat[idx].mean() - ctrl[idx].mean())
lo, hi = np.percentile(bs, [2.5, 97.5])
print(f"反事实提升估计 = {uplift:+.2%}，95%CI [{lo:+.2%}, {hi:+.2%}]，n={n:,}")
print()
print("要多大样本才能稳定分辨 1.5pp 的提升？")
for N in (500, 5000, 50000):
    se = np.sqrt(0.12 * 0.88 / N + 0.135 * 0.865 / N)
    print(f"  n={N:>6,} → 标准误 ≈ {se:.2%}，1.5pp 提升约 {1.5/100/se:.1f} 个标准误"
          f"{'（不够下结论）' if 1.5/100 < 2*se else '（≈可检出）'}")
print()
print("反事实结论必须写清楚：假设（两组可比）、样本量、区间——否则只是故事。")

## 实验 4：40 行的迷你 ReAct——工具、审计、元认知、步数上限

任务："本月利润多少？"。需要 销售额/成本/退款 三个数据，但成本接口今天不可用。
看 Agent 如何：留下可审计轨迹 → 检测到数据缺口 → **拒绝猜测**并报告缺什么。

In [ ]:
import json

DB = {"本月销售额": 120_000}
AVAILABLE = {"查销售额"}                 # "查成本"、"查退款" 接口今天不可用

def agent(question, max_steps=5):
    trace, evidence, missing = [], {}, []
    needed = ["查销售额", "查成本", "查退款"]
    for step, tool in enumerate(needed, 1):
        if step > max_steps:
            trace.append("Thought: 达到步数上限，停止（防死循环）")
            break
        if tool in AVAILABLE:
            evidence["本月销售额"] = DB["本月销售额"]
            trace.append(f"step{step} Thought: 需要{tool} → Action: {tool} → Observation: 120,000")
        else:
            missing.append(tool)
            trace.append(f"step{step} Thought: 需要{tool} → Action: {tool} → Observation: ⚠️ 工具不可用")
    if missing:
        result = {"answer": None, "missing_data": missing,
                  "reason": "利润=销售额−成本−退款，缺必要数据，拒绝猜测"}
    else:
        result = {"answer": evidence["本月销售额"] - 80_000 - 5_000, "missing_data": []}
    return trace, result

trace, result = agent("本月利润多少？")
print("\n".join(trace))
print("最终输出:", json.dumps(result, ensure_ascii=False))
print()
print("四个要件都在：① 工具输出进 evidence（可审计）② 缺数据明说（元认知）")
print("③ 有步数上限（治理）④ answer=None 而不是编一个数（可靠性 > 好看）。")

## 结论

- 投票/加权投票显著优于挑单条最自信的链，且留下可回放证据（实验 1）
- 置信度先验校准（画可靠性图）再用于路由和人工升级阈值（实验 2）
- 反事实提升要报区间和样本量，1.5pp 级别的效应需要万级样本（实验 3）
- 迷你 ReAct 的骨架：Thought→Action→Observation 循环 + 审计 + 缺口上报 + 步数上限（实验 4）

→ 深入阅读：`ima/第5周-Day5-高级推理技巧与Agent推理框架.md`（工具组合推理、Agent 失败处理清单）